# Titanic Passenger Survival Classification

## 1. Introduction & Project Overview

### Project Background
The Titanic disaster is one of the most infamous maritime tragedies in history. On April 15, 1912, the RMS Titanic sank after colliding with an iceberg, resulting in the deaths of 1502 out of 2224 passengers and crew. This project aims to build a machine learning model that can predict passenger survival based on various features such as age, gender, ticket class, and other demographic information.

### Problem Statement
This is a binary classification problem where we need to predict whether a passenger survived (1) or did not survive (0) the Titanic disaster.

### Project Goals
- Clean and preprocess the Titanic dataset
- Perform exploratory data analysis to understand patterns
- Engineer meaningful features from existing data
- Build and evaluate multiple classification models
- Select the best performing model for survival prediction

### Methodology
1. **Data Collection**: Use the Kaggle Titanic dataset
2. **Data Preprocessing**: Handle missing values, encode categorical variables
3. **Exploratory Data Analysis**: Visualize patterns and relationships
4. **Feature Engineering**: Create new meaningful features
5. **Model Building**: Train multiple classification algorithms
6. **Evaluation**: Compare model performance using various metrics

### Expected Outcomes
- A well-performing classification model
- Insights into factors affecting survival
- A reproducible workflow for similar classification problems

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib
import os

# Set style for visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## 2. Dataset Description

### Dataset Overview
The Titanic dataset is one of the most popular datasets for introductory machine learning. It contains information about 891 passengers from the Titanic's maiden voyage.

### Features Description
- **PassengerId**: Unique identifier for each passenger
- **Survived**: Target variable (0 = No, 1 = Yes)
- **Pclass**: Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd)
- **Name**: Passenger's name
- **Sex**: Passenger's gender
- **Age**: Passenger's age in years
- **SibSp**: Number of siblings/spouses aboard
- **Parch**: Number of parents/children aboard
- **Ticket**: Ticket number
- **Fare**: Passenger fare
- **Cabin**: Cabin number
- **Embarked**: Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton)

### Data Source
This dataset is provided by Kaggle for the 'Titanic: Machine Learning from Disaster' competition.

In [ ]:
# Load the dataset
# Note: You'll need to download the Titanic dataset from Kaggle and place it in the data folder
# Dataset URL: https://www.kaggle.com/c/titanic/data

# For now, let's create a placeholder - replace with actual data path
data_path = 'data/titanic.csv'

try:
    df = pd.read_csv(data_path)
    print("Dataset loaded successfully!")
    print(f"Dataset shape: {df.shape}")
except FileNotFoundError:
    print("Please download the Titanic dataset from Kaggle and place it in the 'data' folder.")
    print("Dataset URL: https://www.kaggle.com/c/titanic/data")
    # For demonstration, we'll create a sample structure
    print("\nCreating sample data structure for demonstration...")
    df = pd.DataFrame({
        'PassengerId': range(1, 11),
        'Survived': [0, 1, 1, 1, 0, 0, 0, 1, 1, 0],
        'Pclass': [3, 1, 3, 1, 3, 3, 1, 3, 2, 2],
        'Name': ['Braund', 'Cumings', 'Heikkinen', 'Futrelle', 'Allen', 'Moran', 'McCarthy', 'Palsson', 'Johnson', 'Nasser'],
        'Sex': ['male', 'female', 'female', 'female', 'male', 'male', 'male', 'male', 'female', 'female'],
        'Age': [22, 38, 26, 35, 35, None, 54, 2, 27, 14],
        'SibSp': [1, 1, 0, 1, 0, 0, 0, 3, 0, 1],
        'Parch': [0, 0, 0, 0, 0, 0, 0, 1, 2, 0],
        'Ticket': ['A/5 21171', 'PC 17599', 'STON/O2. 3101282', '113803', '373450', '330877', '17463', '349909', '237736', '237736'],
        'Fare': [7.25, 71.2833, 7.925, 53.1, 8.05, 8.4583, 51.8625, 21.075, 11.1333, 30.0708],
        'Cabin': [None, 'C85', None, 'C123', None, None, 'E46', None, None, None],
        'Embarked': ['S', 'C', 'S', 'S', 'S', 'Q', 'S', 'S', 'S', 'C']
    })

# Display basic information about the dataset
print("\nDataset Info:")
df.info()

print("\nFirst 5 rows:")
df.head()

In [ ]:
# Display basic statistics
print("Dataset Statistics:")
df.describe(include='all')

## 3. Data Cleaning & Preprocessing

In this section, we'll clean the data by:
1. Handling missing values
2. Encoding categorical variables
3. Feature scaling (if needed)

In [ ]:
# Check for missing values
print("Missing Values:")
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
})
print(missing_df[missing_df['Missing Values'] > 0])

In [ ]:
# Handle missing values

# 1. Age - Fill with median based on sex and class
df['Age'] = df.groupby(['Sex', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))

# 2. Embarked - Fill with mode (most common value)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# 3. Cabin - Create a new feature 'Has_Cabin' and fill missing with 'Unknown'
df['Has_Cabin'] = df['Cabin'].notna().astype(int)
df['Cabin'] = df['Cabin'].fillna('Unknown')

print("Missing values after handling:")
print(df.isnull().sum())

In [ ]:
# Encode categorical variables

# 1. Sex - Convert to binary (0=male, 1=female)
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# 2. Embarked - One-hot encoding
df = pd.get_dummies(df, columns=['Embarked'], prefix='Embarked')

# 3. Pclass - Convert to categorical (already numeric, but we'll keep as is for now)

print("Data after encoding categorical variables:")
df.head()

In [ ]:
# Check data types after preprocessing
print("Data types after preprocessing:")
print(df.dtypes)

## 4. Exploratory Data Analysis (EDA)

Let's explore the data to understand patterns and relationships between variables.

In [ ]:
# Set up the figure for multiple plots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Titanic Survival Analysis', fontsize=16, y=1.02)

# 1. Survival rate by gender
sns.barplot(x='Sex', y='Survived', data=df, ax=axes[0, 0])
axes[0, 0].set_title('Survival Rate by Gender')
axes[0, 0].set_xlabel('Gender (0=Male, 1=Female)')
axes[0, 0].set_ylabel('Survival Rate')

# 2. Survival rate by passenger class
sns.barplot(x='Pclass', y='Survived', data=df, ax=axes[0, 1])
axes[0, 1].set_title('Survival Rate by Passenger Class')
axes[0, 1].set_xlabel('Passenger Class')
axes[0, 1].set_ylabel('Survival Rate')

# 3. Age distribution by survival
sns.histplot(data=df, x='Age', hue='Survived', multiple='stack', ax=axes[0, 2])
axes[0, 2].set_title('Age Distribution by Survival')
axes[0, 2].set_xlabel('Age')
axes[0, 2].set_ylabel('Count')

# 4. Fare distribution by survival
sns.histplot(data=df, x='Fare', hue='Survived', multiple='stack', ax=axes[1, 0])
axes[1, 0].set_title('Fare Distribution by Survival')
axes[1, 0].set_xlabel('Fare')
axes[1, 0].set_ylabel('Count')

# 5. Survival rate by embarkation port
embarked_cols = ['Embarked_C', 'Embarked_Q', 'Embarked_S']
embarked_survival = []
for col in embarked_cols:
    if col in df.columns:
        survival_rate = df[df[col] == 1]['Survived'].mean()
        embarked_survival.append(survival_rate)
    else:
        embarked_survival.append(0)

axes[1, 1].bar(['Cherbourg', 'Queenstown', 'Southampton'], embarked_survival)
axes[1, 1].set_title('Survival Rate by Embarkation Port')
axes[1, 1].set_xlabel('Port')
axes[1, 1].set_ylabel('Survival Rate')

# 6. Overall survival count
sns.countplot(x='Survived', data=df, ax=axes[1, 2])
axes[1, 2].set_title('Overall Survival Count')
axes[1, 2].set_xlabel('Survived (0=No, 1=Yes)')
axes[1, 2].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))

# Select only numeric columns for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_cols].corr()

# Create heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f', cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of Titanic Dataset Features', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Additional EDA: Survival by multiple factors
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Survival Analysis by Multiple Factors', fontsize=16)

# Survival by Sex and Class
sns.barplot(x='Pclass', y='Survived', hue='Sex', data=df, ax=axes[0, 0])
axes[0, 0].set_title('Survival by Class and Gender')
axes[0, 0].set_xlabel('Passenger Class')
axes[0, 0].set_ylabel('Survival Rate')

# Age distribution by class and survival
sns.boxplot(x='Pclass', y='Age', hue='Survived', data=df, ax=axes[0, 1])
axes[0, 1].set_title('Age Distribution by Class and Survival')
axes[0, 1].set_xlabel('Passenger Class')
axes[0, 1].set_ylabel('Age')

# Fare by class and survival
sns.boxplot(x='Pclass', y='Fare', hue='Survived', data=df, ax=axes[1, 0])
axes[1, 0].set_title('Fare Distribution by Class and Survival')
axes[1, 0].set_xlabel('Passenger Class')
axes[1, 0].set_ylabel('Fare')

# Family size analysis
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
sns.barplot(x='FamilySize', y='Survived', data=df, ax=axes[1, 1])
axes[1, 1].set_title('Survival Rate by Family Size')
axes[1, 1].set_xlabel('Family Size')
axes[1, 1].set_ylabel('Survival Rate')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Feature Engineering

Let's create new features that might improve our model's performance.

In [ ]:
# Feature Engineering

# 1. Family Size (already created above)
# df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# 2. Is Alone (binary feature)
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# 3. Title extraction from Name
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# 4. Age groups
def categorize_age(age):
    if pd.isna(age):
        return 'Unknown'
    elif age <= 12:
        return 'Child'
    elif age <= 18:
        return 'Teenager'
    elif age <= 60:
        return 'Adult'
    else:
        return 'Senior'

df['AgeGroup'] = df['Age'].apply(categorize_age)

# 5. Fare groups
def categorize_fare(fare):
    if fare <= 7.91:
        return 'Low'
    elif fare <= 14.45:
        return 'Medium-Low'
    elif fare <= 31.0:
        return 'Medium-High'
    else:
        return 'High'

df['FareGroup'] = df['Fare'].apply(categorize_fare)

# 6. Title categories (combine rare titles)
title_mapping = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare',
    'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare', 'Mme': 'Mrs',
    'Capt': 'Rare', 'Sir': 'Rare'
}

df['Title'] = df['Title'].map(title_mapping).fillna('Rare')

print("Feature engineering completed. New features created:")
print("- FamilySize")
print("- IsAlone")
print("- Title")
print("- AgeGroup")
print("- FareGroup")

print("\nValue counts for new features:")
print("\nTitle distribution:")
print(df['Title'].value_counts())
print("\nAge Group distribution:")
print(df['AgeGroup'].value_counts())
print("\nFare Group distribution:")
print(df['FareGroup'].value_counts())

In [ ]:
# Encode the new categorical features

# Encode Title
title_encoder = LabelEncoder()
df['Title_encoded'] = title_encoder.fit_transform(df['Title'])

# Encode AgeGroup
age_group_encoder = LabelEncoder()
df['AgeGroup_encoded'] = age_group_encoder.fit_transform(df['AgeGroup'])

# Encode FareGroup
fare_group_encoder = LabelEncoder()
df['FareGroup_encoded'] = fare_group_encoder.fit_transform(df['FareGroup'])

# One-hot encode categorical features
df = pd.get_dummies(df, columns=['Title', 'AgeGroup', 'FareGroup'], prefix=['Title', 'Age', 'Fare'])

print("Encoding completed. Current dataset shape:", df.shape)
print("\nCurrent columns:")
print(df.columns.tolist())

## 6. Model Building

Now we'll build and train multiple classification models.

In [ ]:
# Prepare data for modeling

# Select features for modeling (exclude non-predictive columns)
exclude_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Survived']
feature_cols = [col for col in df.columns if col not in exclude_cols]

X = df[feature_cols]
y = df['Survived']

print(f"Features selected for modeling: {len(feature_cols)}")
print("Features:", feature_cols)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# Scale numerical features
scaler = StandardScaler()
numerical_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize']
numerical_cols = [col for col in numerical_cols if col in X_train.columns]

if numerical_cols:
    X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
    X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])
    print(f"\nScaled numerical columns: {numerical_cols}")

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100)
}

# Train and evaluate models
model_results = {}
trained_models = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train the model
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Cross-validation score
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    model_results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'CV Mean': cv_mean,
        'CV Std': cv_std
    }
    
    print(f"{name} - Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
    print(f"Cross-validation: {cv_mean:.4f} (+/- {cv_std:.4f})")

In [ ]:
# Feature importance for Random Forest
rf_model = trained_models['Random Forest']
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot feature importance
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(15)
sns.barplot(x='Importance', y='Feature', data=top_features)
plt.title('Top 15 Feature Importance (Random Forest)', fontsize=16)
plt.xlabel('Importance')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

print("Top 10 Most Important Features:")
print(feature_importance.head(10))

## 7. Model Evaluation

Let's evaluate and compare our models using various metrics.

In [ ]:
# Create comparison table
results_df = pd.DataFrame(model_results).T
print("Model Performance Comparison:")
print(results_df.round(4))

# Find best model based on accuracy
best_model_name = results_df['Accuracy'].idxmax()
best_accuracy = results_df['Accuracy'].max()
print(f"\nBest Model: {best_model_name} (Accuracy: {best_accuracy:.4f})")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Model Performance Comparison', fontsize=16)

# Accuracy comparison
results_df['Accuracy'].plot(kind='bar', ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Model Accuracy Comparison')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].tick_params(axis='x', rotation=45)

# Precision comparison
results_df['Precision'].plot(kind='bar', ax=axes[0, 1], color='lightgreen')
axes[0, 1].set_title('Model Precision Comparison')
axes[0, 1].set_ylabel('Precision')
axes[0, 1].tick_params(axis='x', rotation=45)

# Recall comparison
results_df['Recall'].plot(kind='bar', ax=axes[1, 0], color='lightcoral')
axes[1, 0].set_title('Model Recall Comparison')
axes[1, 0].set_ylabel('Recall')
axes[1, 0].tick_params(axis='x', rotation=45)

# F1-Score comparison
results_df['F1-Score'].plot(kind='bar', ax=axes[1, 1], color='gold')
axes[1, 1].set_title('Model F1-Score Comparison')
axes[1, 1].set_ylabel('F1-Score')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices', fontsize=16)

for idx, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(f'{name}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xticklabels(['No', 'Yes'])
    axes[idx].set_yticklabels(['No', 'Yes'])

plt.tight_layout()
plt.show()

In [ ]:
# Detailed classification reports
for name, model in trained_models.items():
    print(f"\n{'='*50}")
    print(f"Classification Report - {name}")
    print(f"{'='*50}")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=['Did Not Survive', 'Survived']))

In [ ]:
# Save the best model
best_model = trained_models[best_model_name]
model_filename = 'models/best_titanic_model.pkl'
joblib.dump(best_model, model_filename)

# Save the scaler
scaler_filename = 'models/scaler.pkl'
joblib.dump(scaler, scaler_filename)

# Save feature columns
feature_columns_filename = 'models/feature_columns.pkl'
joblib.dump(feature_cols, feature_columns_filename)

print(f"Best model ({best_model_name}) saved as: {model_filename}")
print(f"Scaler saved as: {scaler_filename}")
print(f"Feature columns saved as: {feature_columns_filename}")

## 8. Conclusion & Future Improvements

### Project Summary
This project successfully built a machine learning model to predict Titanic passenger survival. We implemented a complete data science workflow including data cleaning, exploratory analysis, feature engineering, and model evaluation.

In [ ]:
# Final summary
print("="*60)
print("TITANIC SURVIVAL CLASSIFICATION - PROJECT SUMMARY")
print("="*60)

print(f"\nDataset Shape: {df.shape}")
print(f"Features Used: {len(feature_cols)}")
print(f"Best Model: {best_model_name}")
print(f"Best Accuracy: {best_accuracy:.4f}")

print("\nKey Findings:")
print("1. Gender was a strong predictor - females had higher survival rates")
print("2. Passenger class significantly affected survival outcomes")
print("3. Age and family size also played important roles")
print("4. Feature engineering improved model performance")

print("\nModel Performance Summary:")
for model_name, metrics in model_results.items():
    print(f"{model_name}: {metrics['Accuracy']:.4f} accuracy")

### Key Insights

1. **Demographics Matter**: Female passengers had significantly higher survival rates than males
2. **Class Influence**: First-class passengers had better survival rates than third-class passengers
3. **Age Factor**: Children and young adults had higher survival rates
4. **Family Dynamics**: Small families had better survival rates than large families or individuals traveling alone

### Model Performance

The best performing model achieved good accuracy in predicting passenger survival. The Random Forest model provided good interpretability through feature importance, while Gradient Boosting offered competitive performance.

### Future Improvements

1. **Advanced Feature Engineering**:
   - Extract more information from ticket numbers
   - Analyze cabin locations more deeply
   - Create interaction features between variables

2. **Model Optimization**:
   - Hyperparameter tuning using GridSearchCV
   - Try more advanced algorithms (XGBoost, LightGBM)
   - Implement ensemble methods

3. **Data Augmentation**:
   - Use external historical data
   - Incorporate crew member information
   - Add weather and ship condition data

4. **Advanced Techniques**:
   - Implement cross-validation with stratification
   - Use feature selection techniques
   - Apply dimensionality reduction (PCA)

5. **Deployment Considerations**:
   - Create a web interface for predictions
   - Implement model monitoring and retraining
   - Add confidence intervals to predictions

### Learning Outcomes

This project demonstrates:
- Complete machine learning workflow implementation
- Data preprocessing and feature engineering techniques
- Multiple classification algorithms comparison
- Model evaluation and selection methods
- Practical application of data science concepts

### References

- Kaggle Titanic Dataset: https://www.kaggle.com/c/titanic
- Scikit-learn Documentation: https://scikit-learn.org/
- Matplotlib Documentation: https://matplotlib.org/
- Seaborn Documentation: https://seaborn.pydata.org/